# Test sub agents

In [1]:
!pip install -r ../requirements.txt

In [2]:
import importlib.util
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, "..")

from langchain_mcp_adapters.client import MultiServerMCPClient
from robot_tools import create_robot_tools
from sensors.image import CAMERA_TOPIC, resolve_camera_topic



mcp_url = os.environ.get("ROS2_MCP_URL", "http://localhost:8000/sse")
mcp_client = MultiServerMCPClient(
    {"ros2_mcp": {"transport": "sse", "url": mcp_url}}
)
tools = await mcp_client.get_tools()
tools_by_name = {tool.name: tool for tool in tools}
required_tools = {"list_topics", "get_image", "topic_echo", "drive_robot"}
missing_tools = required_tools - tools_by_name.keys()
if missing_tools:
    raise RuntimeError(f"Missing MCP tools: {sorted(missing_tools)}")

camera_topic = await resolve_camera_topic(
    tools_by_name["list_topics"], CAMERA_TOPIC
)
robot_tools = create_robot_tools(
    tools_by_name["topic_echo"],
    tools_by_name["get_image"],
    camera_topic,
)


controller_spec = importlib.util.spec_from_file_location(
    "raspbot_agentic_controller", "../ros2-agentic-ai.py"
)
if controller_spec is None or controller_spec.loader is None:
    raise RuntimeError("Could not load ros2-agentic-ai.py.")
controller = importlib.util.module_from_spec(controller_spec)
controller_spec.loader.exec_module(controller)

drive_robot = controller.create_driver(tools_by_name["drive_robot"])
subagents = controller.create_subagents(robot_tools, drive_robot)


test_state = {
    "heading_degrees": 0,
    "scans": 0,
    "distance": None,
    "selected_direction": None,
    "selected_clearance_cm": None,
    "advance_distance_cm": 0.0,
    "photos": [],
    "agent_results": {},
}



## Photographer

In [3]:

try:
    photographer_output = await subagents["photographer"](test_state)
    test_state.update(photographer_output)
    print("Photographer output:")
    print(json.dumps(photographer_output, indent=2, ensure_ascii=False))
finally:
    await drive_robot(
        linear_x=0.0,
        angular_z=0.0,
        duration=controller.STOP_PULSE_SECONDS,
    )


Photographer: saved robot_photos/ros2_photo_20261126T195254_534314Z.png
Photographer output:
{
  "last_agent": "photographer",
  "last_photo_path": "robot_photos/ros2_photo_20261126T195254_534314Z.png",
  "last_photo_at": 122689.555339818,
  "photos": [
    "robot_photos/ros2_photo_20261126T195254_534314Z.png"
  ],
  "agent_results": {
    "photographer": {
      "photo_path": "robot_photos/ros2_photo_20261126T195254_534314Z.png"
    }
  }
}
ROS 2 /cmd_vel vector: linear=(0.0, 0.0, 0.0), angular=(0.0, 0.0, 0.0), duration=0.2s


## Explorer

In [7]:

try:
    explorer_output = await subagents["explorer"](test_state)
    test_state.update(explorer_output)
    print("Explorer output:")
    print(json.dumps(explorer_output, indent=2, ensure_ascii=False))
finally:
    await drive_robot(
        linear_x=0.0,
        angular_z=0.0,
        duration=controller.STOP_PULSE_SECONDS,
    )


Explorer: group 1, direction Z=+0°, clearance 6.8 cm
ROS 2 /cmd_vel vector: linear=(0.0, 0.0, 0.0), angular=(0.0, 0.0, -0.25), duration=0.3490658503988659s
Explorer: group 1, direction Z=-25°, clearance 5.1 cm
ROS 2 /cmd_vel vector: linear=(0.0, 0.0, 0.0), angular=(0.0, 0.0, 0.25), duration=0.6981317007977318s
Explorer: group 1, direction Z=+25°, clearance 5.1 cm
ROS 2 /cmd_vel vector: linear=(0.0, 0.0, 0.0), angular=(0.0, 0.0, 0.25), duration=0.6981317007977318s
Explorer: group 2, direction Z=+75°, clearance 5.1 cm
ROS 2 /cmd_vel vector: linear=(0.0, 0.0, 0.0), angular=(0.0, 0.0, -0.25), duration=1.3962634015954636s
Explorer: group 2, direction Z=-25°, clearance 125.8 cm
Explorer: restoring starting direction (turn +25°).
ROS 2 /cmd_vel vector: linear=(0.0, 0.0, 0.0), angular=(0.0, 0.0, 0.25), duration=0.3490658503988659s
Explorer decision: Z=-25° (125.8 cm)
Explorer output:
{
  "last_agent": "explorer",
  "heading_degrees": 0,
  "readings": [
    {
      "group": 1,
      "direction"

## Advancer

In [5]:

try:
    current_clearance_cm = float(
        await robot_tools["read_proximity"].ainvoke({})
    )
    requested_distance_cm = controller.advance_distance_for_clearance(
        current_clearance_cm,
        obstacle_clearance_cm=controller.MIN_CLEARANCE_CM,
        obstacle_safety_margin_cm=controller.FORWARD_SAFETY_MARGIN_CM,
        advance_fraction=controller.FORWARD_STEP_FRACTION,
        maximum_advance_distance_cm=controller.MAX_ADVANCE_DISTANCE_CM,
    )
    requested_duration = requested_distance_cm / (
        controller.FORWARD_SPEED_MPS * 100
    )
    if (
        current_clearance_cm <= controller.MIN_ADVANCE_DISTANCE_CM
        or requested_duration < controller.MIN_FORWARD_PULSE_SECONDS
    ):
        print(
            "Advancer not called: current clearance "
            f"({current_clearance_cm:.1f} cm) is too small."
        )
    else:
        test_state["advance_distance_cm"] = requested_distance_cm
        advancer_output = await subagents["advancer"](test_state)
        test_state.update(advancer_output)
        print("Advancer output:")
        print(json.dumps(advancer_output, indent=2, ensure_ascii=False))
finally:
    await drive_robot(
        linear_x=0.0,
        angular_z=0.0,
        duration=controller.STOP_PULSE_SECONDS,
    )


Advancer: moving forward 10.0 cm at 0.050 m/s for 2.00 s.
ROS 2 /cmd_vel vector: linear=(0.05, 0.0, 0.0), angular=(0.0, 0.0, 0.0), duration=2.0s
Advancer output:
{
  "last_agent": "advancer",
  "last_advance_distance_cm": 10.0,
  "last_advance_duration_seconds": 2.0,
  "agent_results": {
    "photographer": {
      "photo_path": "robot_photos/ros2_photo_20261126T195254_534314Z.png"
    },
    "explorer": {
      "readings": [
        {
          "group": 1,
          "direction": "Z=+0°",
          "angle_degrees": 0,
          "distance_cm": 86.69999694824219
        },
        {
          "group": 1,
          "direction": "Z=-25°",
          "angle_degrees": -25,
          "distance_cm": 86.69999694824219
        },
        {
          "group": 1,
          "direction": "Z=+25°",
          "angle_degrees": 25,
          "distance_cm": 90.0999984741211
        }
      ],
      "selected_direction": "Z=+25°",
      "selected_clearance_cm": 90.0999984741211
    },
    "advancer": {
   